# Model Training Module

In [ ]:
import argparse
from pathlib import Path

## train_baselines

In [ ]:
def train_baselines(config_path: str = "config.yaml"):
    """
    Train baseline machine learning models.

    Args:
        config_path: Path to configuration file
    """
    logger.info("Training baseline ML models (Logistic Regression, Random Forest, MLP)")

    evaluator = BaselineModelEvaluator(config_path)
    results = evaluator.train_all_models(cohorts=["ROSMAP", "MSBB", "MAYO"])

    logger.info("Baseline model training complete")
    logger.info(f"Results saved to {evaluator.results_dir}/baseline_metrics.json")

    return results


## train_gnn

In [ ]:
def train_gnn(config_path: str = "config.yaml"):
    """
    Train Graph Attention Network models.

    Args:
        config_path: Path to configuration file
    """
    logger.info("Training Graph Attention Network models")

    from src.models.gnn_model import GNNTrainer

    trainer = GNNTrainer(config_path)
    results = trainer.train(cohorts=["ROSMAP", "MSBB", "MAYO"])

    logger.info("GNN model training complete")
    return results


## explain_gnn

In [ ]:
def explain_gnn(config_path: str = "config.yaml"):
    """
    Run explainability analysis on trained GNN models.

    Args:
        config_path: Path to configuration file
    """
    logger.info("Running explainability analysis on GNN models")

    from src.explain.explain_model import explain_model

    results = explain_model(config_path)

    logger.info("Explainability analysis complete")
    return results


## analyze_stability

In [ ]:
def analyze_stability(config_path: str = "config.yaml"):
    """
    Run ranking stability analysis on explainability results.

    Args:
        config_path: Path to configuration file
    """
    logger.info("Running ranking stability analysis")

    from src.analysis.stability import run_stability_analysis

    results = run_stability_analysis(config_path)

    logger.info("Ranking stability analysis complete")
    return results


## rank_proteins

In [ ]:
def rank_proteins(config_path: str = "config.yaml"):
    """
    Run protein prioritization ranking.

    Args:
    config_path: Path to configuration file
    """
    logger.info("Running protein prioritization ranking")

    from src.analysis.final_ranking import run_protein_prioritization

    results = run_protein_prioritization(config_path)

    logger.info("Protein ranking complete")
    return results



## train_model

In [ ]:
def train_model(config_path: str = "config.yaml"):
    """
    Train models (baseline ML models and GNN) and run full analysis pipeline.

    Args:
    config_path: Path to configuration file
    """
    config = load_config(config_path)
    processed_dir = Path(config["data"]["processed_dir"])
    checkpoint_dir = Path(config["logging"]["checkpoint_dir"])
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    logger.info(f"Training models on data from {processed_dir}")
    logger.info(f"Checkpoint directory: {checkpoint_dir}")
    logger.info(f"Model architecture: {config['model']['architecture']}")

    # Train baseline models first
    train_baselines(config_path)

    # Train GNN models
    train_gnn(config_path)

    # Run explainability analysis
    explain_gnn(config_path)

    # Run ranking stability analysis
    analyze_stability(config_path)

    # Run protein prioritization ranking
    rank_proteins(config_path)

    logger.info("Full model training and analysis pipeline complete")



## Main Execution

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Train models and run analysis")
    parser.add_argument("--config", default="config.yaml", help="Config file path")
    parser.add_argument("--baseline-only", action="store_true", help="Train only baseline models")
    parser.add_argument("--gnn-only", action="store_true", help="Train only GNN models")
    parser.add_argument("--explain-only", action="store_true", help="Run only explainability analysis")
    parser.add_argument("--stability-only", action="store_true", help="Run only stability analysis")
    parser.add_argument("--rank-only", action="store_true", help="Run only protein ranking")
    args = parser.parse_args()

    if args.baseline_only:
        train_baselines(args.config)
    elif args.gnn_only:
        train_gnn(args.config)
    elif args.explain_only:
        explain_gnn(args.config)
    elif args.stability_only:
        analyze_stability(args.config)
    elif args.rank_only:
        rank_proteins(args.config)
    else:
        train_model(args.config)
